# Soil moisture mapping

**Yuxi Zhang** · Oct 25, 2026 · [@yuxi-research](https://github.com/yuxi-research)

📘 This notebook converts CRNS neutron counts to soil moisture.

⚙️ **Main Processing Steps (Calibration)**  



💬 **Notes**

📥 **Input**  

- `./data_lv2/{farm_name}/`
  - `kriged_ok/`  
  - `kriged_b2pk/`
  - `kriged_p2pk/`  

📤 **Output**  
- `./data_lv3/{farm_name}/mapping/`


In [ ]:
%load_ext autoreload
%autoreload 2

import os
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import geopandas as gpd
# from shapely.geometry import Point
import rasterio
import re
# import seaborn as sns
import xarray as xr
import rioxarray as rxr
import warnings
warnings.filterwarnings("ignore")

from figure_settings import update_plot_settings
from utils.calc_metrics import calc_lccc, calc_corr
from utils.rover_process import extract_raster_values
update_plot_settings('xxx-large')
save_pdf = True


In [ ]:
# Path settings
from global_settings import *
farm_name = os.environ.get("FARM_NAME", "Amaroo")
dirs = ProjectPaths(farm_name=farm_name)
proj_crs = dirs.proj_crs
inset_loc = dirs.inset_loc
buffer_dis = BUFFER_DIS


In [ ]:
# Parameters
try:
    mode
except NameError:
    mode = 'b2pk'

In [ ]:
suffix = f'_{mode}'

dir_ok = dirs.lv2_ok
dir_krig = dirs.lv2_krig / f'kriged_{mode}'
dir_cal = dirs.lv2_cal / f'cal_{mode}_clean'
dir_out = dirs.lv3_map / f'maps_{mode}'

dir_out_f = dir_out / "figures"
dir_out_f.mkdir(parents=True, exist_ok=True)

## Read all data
### Find kriged rover counts

In [ ]:
# Find all corrected counts files and dates
regex = re.compile(rf'{farm_name}_(.*)_CountsCorr_grid_clipped{suffix}.tif')
filenames_rover = [f for f in os.listdir(dir_krig) if regex.match(f)]
filenames_rover.sort()
dates_str = [regex.match(f).group(1) for f in filenames_rover]
filenames_rover

In [ ]:
# Parameters
try:
    date_str
except NameError:
    date_str = dates_str[0]
print(f'Processing date: {date_str}')

In [ ]:
filename = [f for f in filenames_rover if date_str in f][0]
date = dt.datetime.strptime(date_str, '%Y-%m-%d')
farm_name, date_str = re.split( '_', filename)[0:2]
print(f'Processing file: {filename} for date: {date_str}')

### Read files

#### Boundary

In [ ]:
# Read boundary
bound = gpd.read_file(os.path.join(dirs.lv1_shp, f'{farm_name}_boundary_epsg{proj_crs}.shp'))
bound = bound.to_crs(proj_crs)

#### Calibrated N0

In [ ]:
file_cal = os.path.join(dir_cal, f'{farm_name}_N0_calibrated{suffix}.csv')
df_cal = pd.read_csv(file_cal, index_col='CalibrationDate')
N0 = df_cal.loc[date_str, 'N0_optimized']

In [ ]:
df_cal

In [ ]:
N0

#### Read soil moisture data

In [ ]:
# Read soil moisture data with covariates
df_topsoil = pd.read_csv(os.path.join(dirs.lv2_data, f'{farm_name}_soilsamples_weighted.csv'), parse_dates=['Date'])
df_topsoil['Date'] = pd.to_datetime(df_topsoil['Date'])
df_topsoil['geometry'] = gpd.points_from_xy(df_topsoil['Longitude'], df_topsoil['Latitude'], crs= '4326').to_crs(proj_crs)
gdf_topsoil = gpd.GeoDataFrame(df_topsoil, geometry='geometry')

# Select soil covariates for the date
gdf_topsoil_date = gdf_topsoil[gdf_topsoil['Date'] == date]
gdf_topsoil_date = gdf_topsoil_date.dropna(subset=['Longitude', 'Latitude'])
print(f'Number of soil moisture samples for {date_str}: {len(gdf_topsoil_date)}')

### Corrected neutron counts at original locations

In [ ]:
filename_rover_orig = os.path.join(dirs.lv2_data, f"{farm_name}_{date_str}_counts_corr.csv")
df_rover_orig = pd.read_csv(filename_rover_orig,)

In [ ]:
df_rover_orig['geometry'] = gpd.points_from_xy(df_rover_orig['LongDec'], df_rover_orig['LatDec'], crs= '4326').to_crs(proj_crs)
gdf_rover_orig = gpd.GeoDataFrame(df_rover_orig, geometry='geometry')
# Check the speed statistics in m/s
gdf_rover_orig['Speed_kmh'].describe() * 1000/3600  # Convert km/h to m/s

In [ ]:
gdf_rover_orig.columns

### Read counts kriged at soil sampling locations

In [ ]:
df_counts_points = os.path.join(dir_krig, f'{farm_name}_{date_str}_CountsCorr_points{suffix}.csv')
if os.path.exists(df_counts_points):
    df_points = pd.read_csv(df_counts_points)
else:
    print(f'File not found: {df_counts_points}')

In [ ]:
block_max = df_points['block_max'].unique()[0]
block_min = df_points['block_min'].unique()[0]
block_max, block_min

#### Read kriged soil covariance

In [ ]:
# Read interploated map of soil covariates
ds_dict = {}
for col in ['SMGrav', 'watLatticeGrav', 'watHydrogenGrav', 'watCarbonGrav', 'SMVol', 'watLatticeVol', 'watHydrogenVol', 'watCarbonVol', 'BD']:
    try:
        rxrdata = rxr.open_rasterio(os.path.join(dir_ok, f'{farm_name}_{date_str}_{col}_weighted_ok.tif'),
                                    mask_and_scale=True).squeeze()
        rxrdata_masked = rxrdata.rio.clip(bound.geometry.buffer(buffer_dis), drop=True, invert=False, all_touched=True)

        ds_dict[col] = rxrdata_masked
    except rasterio.errors.RasterioIOError:
        print(f'File for {col} not found, skipping.')

ds_data = xr.Dataset(ds_dict)
print('DataSet variable information:', list(ds_data.data_vars))

In [ ]:
if 'SMVol' in ds_data.data_vars:
    var_type = 'Vol'
else:
    var_type = 'Grav'
varname = f'SM{var_type}'

#### Read kriged corrected counts

In [ ]:
rxrdata = rxr.open_rasterio(os.path.join(dir_krig, filename),
                                    mask_and_scale=True).squeeze()

rxrdata_masked = rxrdata.rio.clip(bound.geometry.buffer(buffer_dis), drop=True, invert=False, all_touched=True)

# Make sure the masked data has the same spatial reference and resolution as ds_data.
rxrdata_masked = rxrdata_masked.rio.reproject_match(ds_data)
rxrdata_masked.attrs = ds_data.attrs

ds_data['CountsCorr'] = rxrdata_masked[0]
ds_data['kriging_variance'] = rxrdata_masked[1]
list(ds_data.data_vars)


### Filter the data based on kriging variance and value range

In [ ]:
### Filter the data based on kriging variance and value range
arr_variance = ds_data['kriging_variance'].values.ravel()
arr_variance = arr_variance[~np.isnan(arr_variance)]
var_p95 = np.quantile(arr_variance, 0.95)
print(f'95th percentile of kriging variance: {var_p95}')
variance_threshold = 50
# Probably set to a fixed value instead of the 95th percentile, as the latter can be too high for some dates.
ds_filter_var = ds_data['kriging_variance'] < var_p95
ds_filter_range = (block_min < ds_data['CountsCorr']) & (ds_data['CountsCorr'] < block_max)

ds_filter = ds_filter_var & ds_filter_range

In [ ]:
# If apply filter, uncomment this cell
ds_data = ds_data.where(ds_filter)

# Check the data variables in the dataset
ds_data.data_vars

### Convert soil moisture

In [ ]:
# Function to calcualate soil moisture based on the kriged map of soil properties and corrected counts. Bogena 2013.
def compute_theta(ds, N0, vartype='Grav'):
    if type(ds) == xr.Dataset:
        ds_vars = list(ds.data_vars)
        ds_out = xr.Dataset()
    else:
        ds_vars = ds.columns.tolist()
        ds_out = pd.DataFrame(index=ds.index, columns=[f'SM{vartype}'])


    
    # Latice water content estimation
    if f'watHydrogen{vartype}' in ds_vars:
        watEquivalent = ds[f'watHydrogen{vartype}']
    else:
        watEquivalent = ds[f'watLattice{vartype}'] + ds[f'watCarbon{vartype}']


    # Calculate gravimetric soil moisture. Bogena 2013.
    denominator = (ds['CountsCorr'] / N0) - 0.372
    sm = (0.0808 / denominator) - 0.115 - watEquivalent
    
    # Gravimetric/ volumetric soil moisture estimation
    ds_out[f'SM{vartype}'] = sm

    return ds_out

In [ ]:
# Function to calcualate n0 based on the kriged map of known soil moisture and properties. Bogena 2013.
def compute_n0(ds, vartype='Grav'):
    if type(ds) == xr.Dataset:
        ds_vars = list(ds.data_vars)
        ds_out = xr.Dataset()

    else:
        ds_vars = ds.columns.tolist()
        ds_out = pd.DataFrame(index=ds.index, columns=['N0'])


    # Latice water content estimation
    if f'watHydrogen{vartype}' in ds_vars:
        watEquivalent = ds[f'watHydrogen{vartype}']
    else:
        watEquivalent = ds[f'watLattice{vartype}'] + ds[f'watCarbon{vartype}']


    # Calculate N0. Bogena 2013.
    denominator =  0.0808 /(ds[f'SM{vartype}'] + 0.115 + watEquivalent)
    N0 = ds['CountsCorr'] / (denominator + 0.372)

    ds_out['N0'] = N0

    return ds_out

#### Process maps

In [ ]:
# Calulate soil moisture and n0 maps
ds_pred = compute_theta(ds_data, N0, vartype=var_type)
# ds_cal = compute_n0(ds_data, vartype=var_type)

In [ ]:
# Remove negative soil moisture values
ds_pred[varname] = ds_pred[varname].where(ds_pred[varname] > 0)

In [ ]:
# Save to NetCDF
ds_pred.to_netcdf(os.path.join(dir_out, f'{farm_name}_{date_str}_{varname}{suffix}.nc'))
# ds_cal.to_netcdf(os.path.join(pth_out3_map, f'{farm_name}_{date_str}_N0_calibrated{suffix}.nc'))

In [ ]:
# Product
ds_pred[varname].rio.to_raster(os.path.join(dir_out, f'{farm_name}_{date_str}_{varname}{suffix}.tif'), dtype='float32', nodata=np.nan, driver='GTiff')
ds_data['CountsCorr'].rio.to_raster(os.path.join(dir_out, f'{farm_name}_{date_str}_CountsCorr{suffix}.tif'), dtype='float32', nodata=np.nan, driver='GTiff')

#### Process rover measurement locations

In [ ]:

# Read the kriged map of soil moisture and properties, and extract the values at the rover points.
for ds_var in ds_data.data_vars:
    if ds_var in [f'watLattice{var_type}', f'watHydrogen{var_type}', f'watCarbon{var_type}']:
        gdf_rover_orig[ds_var] = extract_raster_values(gdf_rover_orig.geometry, ds_data[ds_var])
        print(f'Extracted values for {ds_var} at rover points.')

# # Check if the extracted values match the raster values at the rover points
# ds_data[f'watLattice{var_type}'].plot()
# ax = plt.gca()
# gdf_rover_orig.plot(column=f'watLattice{var_type}', markersize=50, ax=ax, edgecolor='grey', legend=False)

In [ ]:
gdf_rover_orig[f'SM{var_type}_pred'] = compute_theta(gdf_rover_orig, N0, vartype=var_type)
gdf_rover_orig.to_csv(os.path.join(dir_out, f'{farm_name}_{date_str}_counts_output_{mode}.csv'), index=False)


## Check mapped soil moisture

In [ ]:
add_contour = True
clabel_kwrds = {"fontsize": 12}

# Read DEM for contour lines
try:
    dem_proj = rxr.open_rasterio(os.path.join(dir_ok, f'{farm_name}_static_DEM_proj.tif'), mask_and_scale=True).squeeze()
    twi_proj = rxr.open_rasterio(os.path.join(dir_ok, f'{farm_name}_static_TWI_proj.tif'), mask_and_scale=True).squeeze()
    dem_clip = dem_proj.rio.clip(bound.geometry.buffer(buffer_dis), drop=True, invert=False, all_touched=True)
except:
    dem_proj = None

# Function: add_contour_to_plot. Add contour lines from a shapefile to an existing plot.
def add_contour_to_plot(ax, da, interval=50, cmap='viridis', linewidths=1, alpha=0.7):
    """
    Add contour lines from a shapefile to an existing plot.
    """
    x = da.x.values
    y = da.y.values
    Z = da.values
    # contour interval
    vmax = np.ceil(np.nanmax(Z) / 100) * 100
    vmin = np.floor(np.nanmin(Z) / 100) * 100

    # Set contour levels based on the data range and specified interval
    minor_levels = np.arange(vmin, vmax+0.1, interval)
    # cs = ax.contour(x, y, Z, levels=minor_levels, cmap=cmap, linewidths=1, alpha=alpha, vmin=vmin, vmax=vmax)
    cs = ax.contour(x, y, Z, levels=minor_levels, colors="#666666", linewidths=1, alpha=alpha, vmin=vmin, vmax=vmax)

    # major_levels = np.arange(vmin, vmax+0.1, interval*2)
    # cs = ax.contour(x, y, Z, levels=major_levels, colors='#3a3a3a', linewidths=1, alpha=alpha, vmin=vmin, vmax=vmax)

    # Add contour labels
    ax.clabel(cs, inline=True, **clabel_kwrds)

if False:
    plt.figure(figsize=(6, 4))
    ax = plt.gca()
    if dem_proj is not None:
        add_contour_to_plot(ax, dem_proj, interval=20)


In [ ]:
N0 = df_cal.loc[date_str, 'N0_optimized']
N0

### Map 1: Predicted and sampled soil moisture

In [ ]:
def reverse_inset_loc(loc):
    if loc == 'upper':
        return 'bottom'
    elif loc == 'lower':
        return 'top'
    elif loc == 'left':
        return 'right'
    elif loc == 'right':
        return 'left' 


inset_loc_xaxis = reverse_inset_loc(inset_loc.split()[0])
inset_loc_yaxis = reverse_inset_loc(inset_loc.split()[1])
inset_fontsize = 14

In [ ]:
## Map 1: Predicted soil moisture and sampled soil moisture
var_plot = varname
var_show = f'SM {var_type}'
unit = r'$\mathrm{m}^3\ \mathrm{m}^{-3}$' if var_type == 'Vol' else r'$\mathrm{g}\ \mathrm{g}^{-1}$'

fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(8, 9), sharex='all', sharey='all')

arr = ds_pred[var_plot].data.ravel()
arr = arr[~np.isnan(arr)]
vmin, vq1, vq99, vmax = np.quantile(arr, [0, 0.01, 0.99, 1])

arr2 = gdf_topsoil_date[f'{var_plot}'].values
arr2 = arr2[~np.isnan(arr2)]
# vmin = 0
# vmax = 0.5
#### Plotting
ds_pred[var_plot].plot(ax=ax, add_colorbar=True, cmap='Spectral', zorder=-10,
             cbar_kwargs={'label': f'{var_show} ({unit})', 'shrink': 0.4}, 
             vmin=vmin, vmax=vq99
            )

# Add contour lines of DEM
if add_contour and dem_proj is not None:
    add_contour_to_plot(ax, dem_proj, interval=10)

gdf_topsoil_plot = gdf_topsoil_date.dropna(subset=[var_plot], how='any')
# Add soil moisture sampling points
gdf_topsoil_plot.plot(ax=ax, column=f'{var_plot}', marker='D', edgecolor='black', zorder=20,
                    cmap='Spectral', legend=False, # alpha=0.3,
                    legend_kwds={'label': f'{var_show} sampled ({unit})', 'shrink': 0.4}, 
                    vmin=vmin, vmax=vq99
                    )


# Annotate the sampling points with their values
for i, row in gdf_topsoil_plot.iterrows():
    ax.annotate(row['SiteID'], (row.geometry.x, row.geometry.y,), fontsize=inset_fontsize,
                xytext=(1, 3), textcoords='offset points', color="#333333", alpha=0.9, zorder=21)

#### Axes settings
# ax.scatter([0,0], [0,1], color='none', marker='s', edgecolor='black', label='Rover Data')
ax.scatter([0,0], [0,1], color='none', marker='D', edgecolor='black', label='Soil Samples')
ax.legend(loc=dict_legend_loc.get(farm_name, 'lower left'), frameon=True, ) # fontsize=inset_fontsize

bound.plot(ax=ax, facecolor='none', edgecolor='black', lw=1)  # Plot boundary
ax.set_xlim(bound.total_bounds[0]-buffer_dis, bound.total_bounds[2]+buffer_dis)
ax.set_ylim(bound.total_bounds[1]-buffer_dis, bound.total_bounds[3]+buffer_dis)
ax.tick_params(direction='in', length=4)  # Set tick parameters
ax.set_xlabel('Easting (m)')  # X label
ax.set_ylabel('Northing (m)')  # Y label
# ax.grid(True, which='both', linestyle='--', linewidth=0.5)
ax.set_aspect('equal', adjustable='box')  # Set aspect ratio to equal
ax.set_title(f'{farm_name} {date_str} {var_show} ({mode.upper()})')
# ax.set_title(f'{farm_name} {date_str}')


#### Add histogram to the upper left corner
if True:
    from mpl_toolkits.axes_grid1.inset_locator import inset_axes
    color = 'deeppink'
    inset_ax = inset_axes(ax, width="30%", height="20%", borderpad=0, loc=inset_loc)
    # Plotting
    inset_ax.hist(arr, bins=20, color=color, alpha=1, edgecolor='white', lw=0.2, density=True)
    # inset_ax.hist(arr2, bins=20, color='green', alpha=1, edgecolor='white', lw=0.2, density=True)

    inset_ax.set_xlim(vmin, vmax)  # Set x-axis limits to be the same as the main plot
    # Set tick parameters
    inset_ax.tick_params(labelsize=inset_fontsize, direction='in', length=2, colors=color)  # Set tick parameters
    # Adjust the position of the y-axis label to right
    inset_ax.yaxis.set_ticks_position(inset_loc_yaxis)  # Y ticks on right
    inset_ax.yaxis.set_label_position(inset_loc_yaxis)  # Y label on to right
    inset_ax.xaxis.set_ticks_position(inset_loc_xaxis)  # Y ticks on right
    inset_ax.xaxis.set_label_position(inset_loc_xaxis)  # Y label on to right
    # Set spine colors
    inset_ax.spines[inset_loc_xaxis].set_color(color)
    inset_ax.spines[inset_loc_yaxis].set_color(color)
    # Set labels and title
    inset_ax.set_xlabel(var_show, color=color, fontsize=inset_fontsize, labelpad=1)  # X label
    # inset_ax.set_ylabel("Frequency", color=color, fontsize=inset_fontsize, labelpad=1)  # Y label
    inset_ax.set_ylabel("")  # Y label
    inset_ax.set_yticks([])  # Remove y ticks
    # inset_ax.set_title('Histogram', fontsize=inset_fontsize, pad=2, color=color)
    # Gridline
    inset_ax.grid(True, which='both', linestyle='', linewidth=0.1)
    # inset_ax.legend(fontsize=inset_fontsize, loc=2)

# plt.suptitle("")
plt.tight_layout()
plt.savefig(os.path.join(dir_out_f, f'{farm_name}_{date_str}_{var_plot}{suffix}.png'))
if save_pdf:
    plt.savefig(os.path.join(dir_out_f, f'{farm_name}_{date_str}_{var_plot}{suffix}.pdf'))
plt.show()


In [ ]:
xticks = ax.get_xticks()
yticks = ax.get_yticks()


### Map 2: Kriged neutron counts

In [ ]:
# Read neutron counts

block_support_train = pd.read_csv(os.path.join(dir_krig, f'{farm_name}_{date_str}_CountsCorr_block_support_train{suffix}.csv'))
block_support_test = pd.read_csv(os.path.join(dir_krig, f'{farm_name}_{date_str}_CountsCorr_block_support_test{suffix}.csv'))
""
block_support_train['geometry'] = gpd.points_from_xy(block_support_train['x'], block_support_train['y'], crs=proj_crs)
block_support_test['geometry'] = gpd.points_from_xy(block_support_test['x'], block_support_test['y'], crs=proj_crs)
block_support_train = gpd.GeoDataFrame(block_support_train, geometry='geometry')
block_support_test = gpd.GeoDataFrame(block_support_test, geometry='geometry')

In [ ]:
## Map: Kriged counts and block-support samples
var_plot = 'CountsCorr'
var_show = 'Corr. Counts'
unit = r'$\mathrm{cpm}$'
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(8, 9), sharex='all', sharey='all')

arr = ds_data[var_plot].data.ravel()
arr = arr[~np.isnan(arr)]
vmin, vq1, vq99, vmax = np.quantile(arr, [0, 0.01, 0.99, 1])

#### Plotting
ds_data[var_plot].plot(ax=ax, add_colorbar=True, cmap='Spectral_r', zorder=-10,
             cbar_kwargs={'label': var_show + f' ({unit})', 'shrink': 0.4}, 
             vmin=vmin, vmax=vq99
            )


block_support_test.plot(ax=ax, column=f'{var_plot}_kriged', marker='s', edgecolor='black',
                    cmap='Spectral_r', legend=False, # alpha=0.3,
                    legend_kwds={'label': var_show+'_sample', 'shrink': 0.4}, 
                    vmin=vmin, vmax=vq99
                    )

block_support_train.plot(ax=ax, column=f'{var_plot}_kriged', marker='o', edgecolor='black',
                    cmap='Spectral_r', legend=False, # alpha=0.3,
                    legend_kwds={'label': var_show+'_sample', 'shrink': 0.4}, 
                    vmin=vmin, vmax=vq99
                    )

#### Axes settings

ax.scatter([0,0], [0,1], color='none', marker='o', edgecolor='black', label='Interpolation set')
ax.scatter([0,0], [0,1], color='none', marker='s', edgecolor='black', label='Evaluation set')
# ax.legend(loc=1, bbox_to_anchor=(0.65, 1), frameon=False) # Upper center
ax.legend(loc=dict_legend_loc.get(farm_name, 'lower left'), frameon=True, ) # fontsize=inset_fontsize

bound.plot(ax=ax, facecolor='none', edgecolor='black', lw=1)  # Plot boundary
ax.set_xlim(bound.total_bounds[0]-buffer_dis, bound.total_bounds[2]+buffer_dis)
ax.set_ylim(bound.total_bounds[1]-buffer_dis, bound.total_bounds[3]+buffer_dis)
ax.tick_params(direction='in', length=4)  # Set tick parameters
ax.set_xlabel('Easting (m)')  # X label
ax.set_ylabel('Northing (m)')  # Y label
# ax.grid(True, which='both', linestyle='--', linewidth=0.5)
ax.set_aspect('equal', adjustable='box')  # Set aspect ratio to equal

ax.set_title(f'{farm_name} {date_str} {var_show} ({mode.upper()})')
# ax.set_title(f'{farm_name} {date_str}')

#### Add histogram to the upper left corner
if True:
    from mpl_toolkits.axes_grid1.inset_locator import inset_axes
    color = 'deeppink'
    inset_ax = inset_axes(ax, width="30%", height="20%", borderpad=0, loc=inset_loc)
    # Plotting
    inset_ax.hist(arr, bins=20, color=color, alpha=1, edgecolor='white', lw=0.2, density=True)

    inset_ax.set_xlim(vmin, vmax)  # Set x-axis limits to be the same as the main plot
    # Set tick parameters
    inset_ax.tick_params(labelsize=inset_fontsize, direction='in', length=2, colors=color)  # Set tick parameters
    # Adjust the position of the y-axis label to right
    inset_ax.yaxis.set_ticks_position(inset_loc_yaxis)  # Y ticks on right
    inset_ax.yaxis.set_label_position(inset_loc_yaxis)  # Y label on to right
    inset_ax.xaxis.set_ticks_position(inset_loc_xaxis)  # Y ticks on right
    inset_ax.xaxis.set_label_position(inset_loc_xaxis)  # Y label on to right
    # Set spine colors
    inset_ax.spines[inset_loc_xaxis].set_color(color)
    inset_ax.spines[inset_loc_yaxis].set_color(color)
    # Set labels and title
    inset_ax.set_xlabel(var_show, color=color, fontsize=inset_fontsize, labelpad=1)  # X label
    # inset_ax.set_ylabel("Frequency", color=color, fontsize=inset_fontsize, labelpad=1)  # Y label
    inset_ax.set_ylabel("")  # Y label
    inset_ax.set_yticks([])  # Remove y ticks
    # inset_ax.set_title('Histogram', fontsize=inset_fontsize, pad=2, color=color)
    # Gridline
    inset_ax.grid(True, which='both', linestyle='', linewidth=0.1)
    # inset_ax.legend(fontsize=inset_fontsize, loc=2)

# plt.suptitle("")
plt.tight_layout()
plt.savefig(os.path.join(dir_out_f, f'{farm_name}_{date_str}_{var_plot}{suffix}.png'))
if save_pdf:
    plt.savefig(os.path.join(dir_out_f, f'{farm_name}_{date_str}_{var_plot}{suffix}.pdf'))
plt.show()



In [ ]:
## Map: Kriging variance
var_plot = 'kriging_variance'
var_show = 'Kriging Variance'
unit = r'$\mathrm{cpm}^2$'
fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(8, 9), sharex='all', sharey='all')

arr = ds_data[var_plot].data.ravel()
arr = arr[~np.isnan(arr)]
vmin, vq1, vq99, vmax = np.quantile(arr, [0, 0.01, 0.99, 1])

#### Plotting
ds_data[var_plot].plot(ax=ax, add_colorbar=True, cmap='Spectral_r', zorder=-10,
             cbar_kwargs={'label': f'{var_show} ({unit})', 'shrink': 0.4}, 
             vmin=vmin, vmax=vq99
            )

block_support_test.plot(ax=ax, column=var_plot, marker='s', edgecolor='black',
                    cmap='Spectral_r', legend=False, # alpha=0.3,
                    legend_kwds={'label': f'{var_show}_rover ({unit})', 'shrink': 0.4}, 
                    vmin=vmin, vmax=vq99
                    )

block_support_train.plot(ax=ax, column=var_plot, marker='o', edgecolor='black',
                    cmap='Spectral_r', legend=False, # alpha=0.3,
                    legend_kwds={'label': f'{var_show} ({unit})', 'shrink': 0.4}, 
                    vmin=vmin, vmax=vq99
                    )

#### Axes settings

ax.scatter([0,0], [0,1], color='none', marker='o', edgecolor='black', label='Interpolation set')
ax.scatter([0,0], [0,1], color='none', marker='s', edgecolor='black', label='Evaluation set')
# ax.legend(loc=1, bbox_to_anchor=(0.65, 1), frameon=False) # Upper center
ax.legend(loc=dict_legend_loc.get(farm_name, 'lower left'), frameon=True, ) # fontsize=inset_fontsize

bound.plot(ax=ax, facecolor='none', edgecolor='black', lw=1)  # Plot boundary
ax.set_xticks(xticks)
ax.set_yticks(yticks)
ax.set_xlim(bound.total_bounds[0]-buffer_dis, bound.total_bounds[2]+buffer_dis)
ax.set_ylim(bound.total_bounds[1]-buffer_dis, bound.total_bounds[3]+buffer_dis)
ax.tick_params(direction='in', length=4)  # Set tick parameters
ax.set_xlabel('Easting (m)')  # X label
ax.set_ylabel('Northing (m)')  # Y label
# ax.grid(True, which='both', linestyle='--', linewidth=0.5)
ax.set_aspect('equal', adjustable='box')  # Set aspect ratio to equal
ax.set_title(f'{farm_name} {date_str} {var_show} ({mode.upper()})')

#### Add histogram to the upper left corner
if True:
    from mpl_toolkits.axes_grid1.inset_locator import inset_axes
    color = 'deeppink'
    inset_ax = inset_axes(ax, width="30%", height="20%", borderpad=0, loc=inset_loc)
    # Plotting
    inset_ax.hist(arr, bins=20, color=color, alpha=1, edgecolor='white', lw=0.2, density=True)

    inset_ax.set_xlim(vmin, vmax)  # Set x-axis limits to be the same as the main plot
    # Set tick parameters
    inset_ax.tick_params(labelsize=inset_fontsize, direction='in', length=2, colors=color)  # Set tick parameters
    # Adjust the position of the y-axis label to right
    inset_ax.yaxis.set_ticks_position(inset_loc_yaxis)  # Y ticks on right
    inset_ax.yaxis.set_label_position(inset_loc_yaxis)  # Y label on to right
    inset_ax.xaxis.set_ticks_position(inset_loc_xaxis)  # Y ticks on right
    inset_ax.xaxis.set_label_position(inset_loc_xaxis)  # Y label on to right
    # Set spine colors
    inset_ax.spines[inset_loc_xaxis].set_color(color)
    inset_ax.spines[inset_loc_yaxis].set_color(color)
    # Set labels and title
    inset_ax.set_xlabel(var_show, color=color, fontsize=inset_fontsize, labelpad=1)  # X label
    # inset_ax.set_ylabel("Frequency", color=color, fontsize=inset_fontsize, labelpad=1)  # Y label
    inset_ax.set_ylabel("")  # Y label
    inset_ax.set_yticks([])  # Remove y ticks
    # inset_ax.set_title('Histogram', fontsize=inset_fontsize, pad=2, color=color)
    # Gridline
    inset_ax.grid(True, which='both', linestyle='', linewidth=0.1)
    # inset_ax.legend(fontsize=inset_fontsize, loc=2)

# plt.suptitle("")
plt.tight_layout()
plt.savefig(os.path.join(dir_out_f, f'{farm_name}_{date_str}_{var_plot}{suffix}.png'))
if save_pdf:
    plt.savefig(os.path.join(dir_out_f, f'{farm_name}_{date_str}_{var_plot}{suffix}.pdf'))
plt.show()



### Scatter plot between kriged and measured counts


In [ ]:
var_plot = 'CountsCorr'
var_show = 'Corr. Counts'
unit = 'cpm'

block_support_train_clean = block_support_train.dropna(subset=[var_plot, f'{var_plot}_kriged'], how='any')
block_support_test_clean = block_support_test.dropna(subset=[var_plot, f'{var_plot}_kriged'], how='any')
vmin = min(block_support_train_clean[var_plot].min(), block_support_test_clean[var_plot].min())
vmax = max(block_support_train_clean[var_plot].max(), block_support_test_clean[var_plot].max())

fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(6.5, 6.5), sharex='all', sharey='all')

ax.errorbar(block_support_train_clean[var_plot], block_support_train_clean[f'{var_plot}_kriged'], 
                # yerr=np.sqrt(block_support_train_clean['kriging_variance']), 
                fmt='o', ms=10, c='green', alpha=0.5, label='Interpolation set')
ax.errorbar(block_support_test_clean[var_plot], block_support_test_clean[f'{var_plot}_kriged'], 
                # yerr=np.sqrt(block_support_test_clean['kriging_variance']),
                fmt='s', ms=10, c='deeppink', alpha=0.5, label='Evaluation set')

r_train = calc_corr(block_support_train_clean[var_plot], block_support_train_clean[f'{var_plot}_kriged'])
r_test = calc_corr(block_support_test_clean[var_plot], block_support_test_clean[f'{var_plot}_kriged'])
lccc_train = calc_lccc(block_support_train_clean[var_plot], block_support_train_clean[f'{var_plot}_kriged'])
lccc_test = calc_lccc(block_support_test_clean[var_plot], block_support_test_clean[f'{var_plot}_kriged'])
rmse_train = np.sqrt(np.mean((block_support_train_clean[var_plot] - block_support_train_clean[f'{var_plot}_kriged'])**2))
rmse_test = np.sqrt(np.mean((block_support_test_clean[var_plot] - block_support_test_clean[f'{var_plot}_kriged'])**2))
N_train = len(block_support_train_clean)
N_test = len(block_support_test_clean)

ax = plt.gca()
ax.set_aspect('equal', adjustable='box')

fig_text = '\n'.join([f'Interpolation set (n={N_train}):\nRMSE={rmse_train:.2f}, LCCC={lccc_train:.2f}',
                        f'Evaluation set (n={N_test}):\nRMSE={rmse_test:.2f}, LCCC={lccc_test:.2f}',])


ax.text(0.25, 0.05, fig_text,
        transform=ax.transAxes, color='black', verticalalignment='bottom', horizontalalignment='left',
        bbox=dict(facecolor='none', alpha=0.1, edgecolor='black', boxstyle='square,pad=0.5'))

ax.set_title(f'{farm_name} {date_str} ({mode.upper()})')

# plt.title(fig_text)
plt.xlabel(f'Sampled {var_show} ({unit})')
plt.ylabel(f'Kriged {var_show} ({unit})')
plt.plot([vmin, vmax], [vmin, vmax], ls='--', c='grey')
ax.set_xlim(vmin-0.2*(vmax-vmin), vmax+0.2*(vmax-vmin))
ax.set_ylim(vmin-0.2*(vmax-vmin), vmax+0.2*(vmax-vmin))
plt.legend(loc='upper left')
plt.tight_layout()
plt.savefig(os.path.join(dir_out_f, f'{farm_name}_{date_str}_{var_plot}{suffix}_scatter.png'))
if save_pdf:
        plt.savefig(os.path.join(dir_out_f, f'{farm_name}_{date_str}_{var_plot}{suffix}_scatter.pdf'))

plt.show()


### Other variables

In [ ]:
def plot_map(da_plot, var_plot, var_show=None, unit='', method=mode):
    if var_show is None:
        var_show = var_plot
    fig, ax = plt.subplots(nrows=1, ncols=1, figsize=(8, 9), sharex='all', sharey='all')

    arr = da_plot.data.ravel()
    arr = arr[~np.isnan(arr)]
    arr2 = gdf_topsoil_date[f'{var_plot}'].values
    arr2 = arr2[~np.isnan(arr2)]

    vmin = min(arr.min(), arr2.min())
    vmax = max(arr.max(), arr2.max())

    #### Plotting
    da_plot.plot(ax=ax, add_colorbar=True, cmap='Spectral', zorder=-10,
                cbar_kwargs={'label': f'{var_show} ({unit})', 'shrink': 0.4}, 
                vmin=vmin, vmax=vmax
                )

    # Add soil moisture sampling points
    gdf_topsoil_plot = gdf_topsoil_date.dropna(subset=[var_plot], how='any')
    gdf_topsoil_plot.plot(ax=ax, column=f'{var_plot}', marker='o', markersize=50, edgecolor='black', zorder=20,
                        cmap='Spectral', legend=False, # alpha=0.3,
                        legend_kwds={'label': f'{var_show} sample ({unit})', 'shrink': 0.4}, 
                        vmin=vmin, vmax=vmax
                        )

    # Annotate the sampling points with their values
    for i, row in gdf_topsoil_plot.iterrows():
        ax.annotate(row['SiteID'], (row.geometry.x, row.geometry.y,), fontsize=inset_fontsize,
                    xytext=(1, 3), textcoords='offset points', color='#333333', alpha=0.9, zorder=21)


    #### Axes settings

    bound.plot(ax=ax, facecolor='none', edgecolor='black', lw=1)  # Plot boundary
    ax.set_xlim(bound.total_bounds[0]-buffer_dis, bound.total_bounds[2]+buffer_dis)
    ax.set_ylim(bound.total_bounds[1]-buffer_dis, bound.total_bounds[3]+buffer_dis)
    ax.tick_params(direction='in', length=4)  # Set tick parameters
    ax.set_xlabel('Easting (m)')  # X label
    ax.set_ylabel('Northing (m)')  # Y label
    # ax.grid(True, which='both', linestyle='--', linewidth=0.5)
    ax.set_aspect('equal', adjustable='box')  # Set aspect ratio to equal
    ax.set_title(f'{farm_name} {var_show} ({method})')


    #### Add histogram to the upper left corner
    if True:
        from mpl_toolkits.axes_grid1.inset_locator import inset_axes
        color = 'deeppink'
        inset_ax = inset_axes(ax, width="30%", height="20%", borderpad=0, loc=inset_loc)
        # Plotting
        inset_ax.hist(arr, bins=20, color=color, alpha=1, edgecolor='white', lw=0.2, density=True)

        inset_ax.set_xlim(vmin, vmax)  # Set x-axis limits to be the same as the main plot
        # Set tick parameters
        inset_ax.tick_params(labelsize=inset_fontsize, direction='in', length=2, colors=color)  # Set tick parameters
        # Adjust the position of the y-axis label to right
        inset_ax.yaxis.set_ticks_position(inset_loc_yaxis)  # Y ticks on right
        inset_ax.yaxis.set_label_position(inset_loc_yaxis)  # Y label on to right
        inset_ax.xaxis.set_ticks_position(inset_loc_xaxis)  # Y ticks on right
        inset_ax.xaxis.set_label_position(inset_loc_xaxis)  # Y label on to right
        # Set spine colors
        inset_ax.spines[inset_loc_xaxis].set_color(color)
        inset_ax.spines[inset_loc_yaxis].set_color(color)
        # Set labels and title
        inset_ax.set_xlabel(var_show, color=color, fontsize=inset_fontsize, labelpad=1)  # X label
        # inset_ax.set_ylabel("Frequency", color=color, fontsize=inset_fontsize, labelpad=1)  # Y label
        inset_ax.set_ylabel("")  # Y label
        inset_ax.set_yticks([])  # Remove y ticks
        # inset_ax.set_title('Histogram', fontsize=inset_fontsize, pad=2, color=color)
        # Gridline
        inset_ax.grid(True, which='both', linestyle='', linewidth=0.1)
        # inset_ax.legend(fontsize=inset_fontsize, loc=2)

    # plt.suptitle("")
    plt.tight_layout()
    plt.savefig(os.path.join(dir_out_f, f'{farm_name}_{date_str}_{var_plot}_ok.png'))
    if save_pdf:
        plt.savefig(os.path.join(dir_out_f, f'{farm_name}_{date_str}_{var_plot}_ok.pdf'))
    plt.show()


In [ ]:
## Map: Kriged counts and block-support samples
var_plot = f'watLattice{var_type}'
var_show = f'Lattice Water {var_type}'
unit = r'$\mathrm{m}^3\ \mathrm{m}^{-3}$' if var_type == 'Vol' else r'$\mathrm{g}\ \mathrm{g}^{-1}$'
if var_plot in ds_data.data_vars:
    da_plot = ds_data[var_plot]
    plot_map(da_plot, var_plot, var_show, unit, method='ok')


In [ ]:
var_plot = f'watCarbon{var_type}'
var_show = f'Equil. Wat. Carbon {var_type}'
unit = r'$\mathrm{m}^3\ \mathrm{m}^{-3}$' if var_type == 'Vol' else r'$\mathrm{g}\ \mathrm{g}^{-1}$'
if var_plot in ds_data.data_vars:
    da_plot = ds_data[var_plot]
    plot_map(da_plot, var_plot, var_show, unit, method='ok')


In [ ]:
var_plot = f'watHydrogen{var_type}'
var_show = f'Equil. Wat. Hydrogen {var_type}'
unit = r'$\mathrm{m}^3\ \mathrm{m}^{-3}$' if var_type == 'Vol' else r'$\mathrm{g}\ \mathrm{g}^{-1}$'
if var_plot in ds_data.data_vars:
    da_plot = ds_data[var_plot]
    plot_map(da_plot, var_plot, var_show, unit, method='ok')


In [ ]:
var_plot = 'BD'
var_show = 'Bulk Density'
unit = r'$\mathrm{g}\ \mathrm{cm}^{-3}$'

if var_plot in ds_data.data_vars:
    da_plot = ds_data[var_plot]
    plot_map(da_plot, var_plot, var_show, unit, method='ok')
